# 🐱‍🐉 Clasificación de Pokémon con Transfer Learning

Este script sigue el flujo de trabajo de la **Actividad 3**.

**Objetivo:** Utilizar un modelo pre-entrenado (`EfficientNet`) para clasificar 150 tipos 
diferentes de Pokémon, superando el 80% de precisión.

### 1. Configuración del Entorno
Antes de empezar, preparamos las librerías y detectamos si tenemos GPU (Nvidia o Mac) 
para acelerar el entrenamiento.

In [1]:
import torch
import torchvision
from torch import nn
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import time
from tqdm.auto import tqdm

# Intentamos importar torchinfo para ver resúmenes gráficos del modelo
try:
    from torchinfo import summary
except ImportError:
    print("[INFO] Instalando torchinfo...")
    import subprocess
    subprocess.check_call(["pip", "install", "torchinfo"])
    from torchinfo import summary

# Configuración de dispositivo (Device Agnostic)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps" # Para Mac M1/M2/M3
else:
    device = "cpu"

print(f"[INFO] Usando dispositivo: {device}")

[INFO] Usando dispositivo: cuda


d:\Aplicaciones\Anaconda3\envs\cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.backends.cudnn.benchmark = True

### 2. Preparación de los Datos (El "Chef")

El modelo `EfficientNet` requiere que las imágenes tengan un tamaño y normalización específicos.
Aquí creamos los `DataLoaders` que cortan, procesan y sirven las imágenes en lotes.

In [3]:
# Rutas a los datos (Ajusta estas rutas si tus carpetas están en otro lugar)
# Asumimos que estás en la carpeta 'code' y los datos están en '../data'
train_dir = "../data/train/"
test_dir = "../data/test/"

# 1. Obtener los pesos pre-entrenados de EfficientNet
# "DEFAULT" descarga los mejores pesos disponibles (entrenados en ImageNet)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# 2. Obtener las transformaciones automáticas
# Esto nos dice exactamente cómo el modelo quiere ver las fotos (tamaño, media, desviación)
auto_transforms = weights.transforms()

# 3. Función para crear los cargadores de datos
from torchvision import datasets
from torch.utils.data import DataLoader

def create_dataloaders(train_dir, test_dir, transform, batch_size=64):
    # Cargar carpetas como datasets
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)
    
    # Obtener nombres de las clases
    class_names = train_data.classes

    # Crear DataLoaders
    # shuffle=True en train para que el modelo no memorice el orden
    train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_dataloader, test_dataloader, class_names

# Instanciamos los cargadores
try:
    train_dataloader, test_dataloader, class_names = create_dataloaders(
        train_dir=train_dir,
        test_dir=test_dir,
        transform=auto_transforms
    )
    print(f"[INFO] Clases encontradas: {len(class_names)}")
    print(f"[INFO] Ejemplo de clases: {class_names[:5]}")
except FileNotFoundError:
    print(f"[ERROR] No se encontraron los directorios de datos en {train_dir}")
    print("Por favor verifica la ruta de tus carpetas 'train' y 'test'.")

[INFO] Clases encontradas: 150
[INFO] Ejemplo de clases: ['Abra', 'Aerodactyl', 'Alakazam', 'Alolan Sandslash', 'Arbok']


### 3. Construcción del Modelo (El "Arquitecto")

Esta es la parte clave del **Transfer Learning**.
1. Cargamos el "cerebro" (`features`) que ya sabe ver formas y texturas.
2. Lo "congelamos" para que no olvide lo aprendido.
3. Le cambiamos la "boca" (`classifier`) para que aprenda a decir nombres de Pokémon.

In [4]:
# 1. Cargar modelo base
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# 2. Congelar la base (Feature Extractor)
for param in model.features.parameters():
    param.requires_grad = False

# 3. Reemplazar la cabeza del clasificador
# output_shape debe ser igual al número de tus clases de Pokémon (ej. 150)
output_shape = len(class_names) if 'class_names' in locals() else 150

torch.manual_seed(42)
torch.cuda.manual_seed(42)

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True), 
    torch.nn.Linear(in_features=1280, # Salida fija de EfficientNetB0
                    out_features=output_shape, 
                    bias=True)
).to(device)

# Visualizar estructura del modelo
try:
    summary(model, 
            input_size=(32, 3, 224, 224), 
            col_names=["input_size", "output_size", "num_params", "trainable"])
except Exception as e:
    print(f"[INFO] No se pudo generar el resumen visual: {e}")

### 4. Definición del Motor de Entrenamiento (El "Entrenador")

Aquí definimos las funciones que realizan el trabajo duro:
- `train_step`: Enseña al modelo (calcula error y ajusta pesos).
- `test_step`: Evalúa al modelo (hace examen sin ajustar pesos).
- `train`: El bucle principal que repite esto por varias épocas.

In [5]:
# Definir función de pérdida y optimizador
# CrossEntropyLoss: Mide qué tan mal predice el modelo (combina softmax + log loss)
# Adam: Algoritmo que ajusta los pesos del modelo para reducir el error
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_step(model, dataloader, loss_fn, optimizer, device):
    # Activar modo entrenamiento: habilita Dropout y BatchNorm adaptativos
    model.train()
    train_loss, train_acc = 0, 0
    
    # Itera sobre cada lote (batch) de imágenes del dataset de entrenamiento
    for batch, (X, y) in enumerate(dataloader):
        # Mueve imágenes (X) y etiquetas (y) al dispositivo (GPU o CPU)
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass: predice qué Pokémon es cada imagen
        y_pred = model(X)
        
        # 2. Calcular pérdida: cuánto se equivocó el modelo
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        # 3. Backpropagation: calcula gradientes (derivadas) para ajustar pesos
        optimizer.zero_grad()  # Limpia gradientes anteriores
        loss.backward()  # Calcula nuevos gradientes
        optimizer.step()  # Actualiza los pesos del modelo
        
        # 4. Calcular precisión: qué % de predicciones fueron correctas
        # argmax + softmax: convierte logits en probabilidades y elige la clase más probable
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    
    # Devuelve pérdida y precisión promedio por lote
    return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model, dataloader, loss_fn, device):
    # Activar modo evaluación: desactiva Dropout y congela BatchNorm
    model.eval()
    test_loss, test_acc = 0, 0
    
    # torch.inference_mode: desactiva cálculo de gradientes (ahorra memoria)
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            
            # Hace predicciones SIN ajustar pesos
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Convierte predicciones a clases (sin softmax, solo argmax)
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
    
    # Devuelve pérdida y precisión promedio
    return test_loss / len(dataloader), test_acc / len(dataloader)

def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device):
    # Diccionario para almacenar métricas de cada época
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    
    # Bucle principal: repite el entrenamiento por N épocas
    for epoch in tqdm(range(epochs)):
        # Entrena y obtiene métricas de entrenamiento
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
        # Evalúa y obtiene métricas de prueba (sin ajustar pesos)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fn, device)
        
        # Imprime el progreso de cada época
        print(f"Epoch: {epoch+1} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
        
        # Guarda las métricas en el diccionario
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    
    # Devuelve historial completo de métricas para graficar después
    return results

### 5. Ejecución del Entrenamiento

Ejecutamos el pipeline completo. Si todo está correcto, deberías ver la precisión de prueba (`Test Acc`) 
subiendo rápidamente.

In [ ]:
# Define cuántas vueltas completas hará el modelo sobre TODO el dataset
NUM_EPOCHS = 5 

# Verifica que los dataloaders existan (significa que los datos se cargaron correctamente en el paso 2)
if 'train_dataloader' in locals() and 'test_dataloader' in locals():
    
    # Imprime un mensaje informativo
    print(f"[INFO] Iniciando entrenamiento por {NUM_EPOCHS} épocas...")
    
    # Captura la hora actual para medir tiempo total
    start_time = time.time()
    
    # Ejecuta la función train() que:
    # - Itera por cada época
    # - En cada época: entrena con train_dataloader, evalúa con test_dataloader
    # - Usa optimizer para ajustar los pesos y loss_fn para medir errores
    # - Devuelve un diccionario con las métricas de cada época
    model_results = train(model=model, 
                          train_dataloader=train_dataloader, 
                          test_dataloader=test_dataloader, 
                          optimizer=optimizer, 
                          loss_fn=loss_fn, 
                          epochs=NUM_EPOCHS, 
                          device=device)
    
    # Captura la hora final
    end_time = time.time()
    # Calcula y muestra cuántos segundos tardó el entrenamiento
    print(f"[INFO] Tiempo total de entrenamiento: {end_time-start_time:.2f} segundos")
    
else:
    # Si no existen los dataloaders, muestra error y pide revisar el paso 2
    print("[WARN] No se pueden entrenar los datos porque no se cargaron correctamente (revisa el paso 2).")

[INFO] Iniciando entrenamiento por 5 épocas...


 20%|██        | 1/5 [04:48<19:15, 288.93s/it]

Epoch: 1 | Train Loss: 3.8302 | Train Acc: 0.3676 | Test Loss: 2.3609 | Test Acc: 0.7947


 40%|████      | 2/5 [07:23<10:30, 210.10s/it]

Epoch: 2 | Train Loss: 1.9690 | Train Acc: 0.7841 | Test Loss: 1.2769 | Test Acc: 0.8902


...

 80%|████████  | 4/5 [13:17<03:07, 187.77s/it]

Epoch: 4 | Train Loss: 0.9071 | Train Acc: 0.8934 | Test Loss: 0.6033 | Test Acc: 0.9523


100%|██████████| 5/5 [16:01<00:00, 192.38s/it]

Epoch: 5 | Train Loss: 0.7021 | Train Acc: 0.9174 | Test Loss: 0.4553 | Test Acc: 0.9670
[INFO] Tiempo total de entrenamiento: 961.90 segundos


: 

### 6. Visualización de Resultados
Graficamos las curvas de pérdida y precisión para incluirlas en el reporte.

In [ ]:
def plot_loss_curves(results):
    loss = results['train_loss']
    test_loss = results['test_loss']
    accuracy = results['train_acc']
    test_accuracy = results['test_acc']
    epochs = range(len(results['train_loss']))

    plt.figure(figsize=(15, 7))

    # Graficar pérdida
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='train_loss')
    plt.plot(epochs, test_loss, label='test_loss')
    plt.title('Pérdida (Loss)')
    plt.xlabel('Épocas')
    plt.legend()

    # Graficar precisión
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='train_accuracy')
    plt.plot(epochs, test_accuracy, label='test_accuracy')
    plt.title('Precisión (Accuracy)')
    plt.xlabel('Épocas')
    plt.legend()
    
    plt.show()

if 'model_results' in locals():
    plot_loss_curves(model_results)